# Day 27 — Lowering Toward ExecuTorch, Visualized

ExecuTorch runs models on-device with a tiny C++ runtime. The pipeline starts by lowering
your model to a clean graph with `torch.export` — that graph is what becomes a `.pte`.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3)).eval()
x = torch.randn(3, 4)

ep = torch.export.export(model, (x,))
print(type(ep).__name__)
with torch.no_grad():
    print('matches eager:', torch.allclose(model(x), ep.module()(x), atol=1e-5))

## The exported graph

The exported program holds a backend-agnostic graph of the computation — no Python
control flow, ready for ahead-of-time compilation.

In [ ]:
print(ep.graph_module.code[:800])

## The full ExecuTorch path (conceptual)

```
  nn.Module --torch.export--> ExportedProgram --to_edge/to_executorch--> model.pte
                                                                             |
                                          tiny C++ ExecuTorch runtime  <------+   (on device)
```

Install the runtime with `pip install executorch` (see the README) to emit and run a `.pte`.

**Now build** lower_model / run_exported in `homework.py`, then `pytest -q`.